In [1]:
#1,Quality control -remove blanks
# feb 2, 2026 i have used R package Decontam to statistically remove the blank

In [2]:
# ============================
# STANDARDISE PHYLUM NAMES all verified!
# ============================
rename_map = {
    'actinobacteria':   'actinomycetota',
    'firmicutes':       'bacillota',
    'bacteroidetes':    'bacteroidota',
    'chloroflexi':      'chloroflexota',
    'cyanobacteria':    'cyanobacteriota',
    'proteobacteria':   'pseudomonadota',
    'pseudomonadot':    'pseudomonadota',
    'planctomycetes':   'planctomycetota',
    'gemmatimonadetes': 'gemmatimonadota',
    'verrucomicrobia':  'verrucomicrobiota',
    'acidobacteria':    'acidobacteriota',
    'nitrospirae':      'nitrospirota',
    'chlamydiae':       'chlamydiota',       # add
    'elusimicrobia':    'elusimicrobiota',   # add
    'spirochaetes':     'spirochaetota',     # add
    'tenericutes':      'mycoplasmatota',    # add
}

In [3]:
import pandas as pd

# ============================
# LOAD FILE
# ============================
path = "/content/zotutab_decontam_unstandardized_naming.csv"
df   = pd.read_csv(path, header=None)

print(f"Original shape: {df.shape}")
print(f"First few column names: {df.iloc[1, :5].tolist()}")

# ============================
# IDENTIFY STRUCTURE
# Row 0 = file header info
# Row 1 = sample/taxonomy column names
# Row 2+ = data
# ============================
header_row  = df.iloc[1]
data        = df.iloc[2:].copy()
data.columns = header_row.values
data.reset_index(drop=True, inplace=True)

# ============================
# COMPLETE RENAME MAP
# ============================
rename_map = {
    'actinobacteria':        'actinomycetota',
    'firmicutes':            'bacillota',
    'bacteroidetes':         'bacteroidota',
    'chloroflexi':           'chloroflexota',
    'cyanobacteria':         'cyanobacteriota',
    'proteobacteria':        'pseudomonadota',
    'pseudomonadot':         'pseudomonadota',
    'planctomycetes':        'planctomycetota',
    'gemmatimonadetes':      'gemmatimonadota',
    'verrucomicrobia':       'verrucomicrobiota',
    'acidobacteria':         'acidobacteriota',
    'nitrospirae':           'nitrospirota',
    'chlamydiae':            'chlamydiota',
    'elusimicrobia':         'elusimicrobiota',
    'spirochaetes':          'spirochaetota',
    'tenericutes':           'mycoplasmatota',
    'fusobacteria':          'fusobacteriota',
    'thermodesulfobacteria': 'thermodesulfobacteriota',
    'deinococcus-thermus':   'deinococcota',
    'armatimonadetes':       'armatimonadota',
    'thermotogae':           'thermotogota',
}

# ============================
# APPLY RENAME TO PHYLUM COLUMN
# ============================
# check phylum column exists
if "Phylum" not in data.columns:
    print("ERROR: Phylum column not found")
    print(f"Available columns: {data.columns.tolist()[:10]}")
else:
    # normalise to lowercase, strip whitespace, then remap
    original_phyla  = data["Phylum"].copy()
    data["Phylum"]  = (
        data["Phylum"]
        .str.strip()
        .str.lower()
        .replace(rename_map)
    )

    # ============================
    # AUDIT — show what changed
    # ============================
    changed = original_phyla != data["Phylum"]
    changes = pd.DataFrame({
        "Original": original_phyla[changed],
        "Updated":  data["Phylum"][changed]
    }).drop_duplicates()

    print(f"\nPhylum names updated: {changed.sum()} rows")
    print(f"Unique changes made:  {len(changes)}")
    print()
    print(changes.to_string(index=False))

    # ============================
    # VERIFY NO OLD NAMES REMAIN
    # ============================
    old_names = list(rename_map.keys())
    remaining = data["Phylum"].isin(old_names).sum()
    print(f"\nOld names still present: {remaining} "
          f"(should be 0)")

    # ============================
    # CHECK UNIQUE PHYLA AFTER UPDATE
    # ============================
    print(f"\nUnique phyla after update: "
          f"{data['Phylum'].nunique()}")
    print("\nAll unique phyla:")
    for p in sorted(data["Phylum"].dropna().unique()):
        print(f"  {p}")
        #print(len("Phylum"))

    # ============================
    # SAVE NEW CSV
    # preserving original file structure
    # ============================
    # rebuild full dataframe with original top rows
    out = pd.concat([df.iloc[:2],
                     pd.DataFrame([data.columns.tolist()
                                   if False else None])],
                    ignore_index=True)

    # simpler approach — just save data rows with header
    output_path = "/content/zotutab_decontam_Final_standardized.csv"
    data.to_csv(output_path, index=False)

    print(f"\nSaved: {output_path}")
    print(f"Shape: {data.shape}")

Original shape: (6200, 70)
First few column names: ['Zotu', 'MF.4B', 'MF.5B', 'MF.6B', 'MF.7B']

Phylum names updated: 991 rows
Unique changes made:  18

        Original           Updated
   bacteroidetes      bacteroidota
  proteobacteria    pseudomonadota
   acidobacteria   acidobacteriota
  actinobacteria    actinomycetota
      firmicutes         bacillota
             NaN               NaN
     chloroflexi     chloroflexota
   cyanobacteria   cyanobacteriota
 verrucomicrobia verrucomicrobiota
   pseudomonadot    pseudomonadota
  planctomycetes   planctomycetota
gemmatimonadetes   gemmatimonadota
   elusimicrobia   elusimicrobiota
     nitrospirae      nitrospirota
    fusobacteria    fusobacteriota
    spirochaetes     spirochaetota
     tenericutes    mycoplasmatota
      chlamydiae       chlamydiota

Old names still present: 0 (should be 0)

Unique phyla after update: 49

All unique phyla:
  abditibacteriota
  acidobacteriota
  actinomycetota
  armatimonadota
  atribacterota
  

In [4]:
import pandas as pd

# ============================
# LOAD STANDARDIZED FILE
# ============================
path = "/content/zotutab_decontam_Final_standardized.csv"
df   = pd.read_csv(path)

print(f"Total ZOTUs: {len(df)}")

# ============================
# NORMALISE DOMAIN
# ============================
df["Domain_norm"] = df["Domain"].str.strip().str.lower()
df["Phylum_norm"] = df["Phylum"].str.strip().str.lower()

# ============================
# IDENTIFY NON-PROKARYOTES
# anything that is not bacteria or archaea
# ============================
prokaryote_domains = ["bacteria", "archaea"]
non_prok = df[~df["Domain_norm"].isin(prokaryote_domains)].copy()

print(f"\nNon-prokaryotic ZOTUs: {len(non_prok)}")

# ============================
# PRINT ALL UNIQUE DOMAIN AND
# PHYLUM NAMES FOR NON-PROKARYOTES
# ============================
print("\n" + "="*50)
print("UNIQUE DOMAINS (non-prokaryotic):")
print("="*50)
for d in sorted(non_prok["Domain"].dropna().unique()):
    count = (non_prok["Domain"] == d).sum()
    print(f"  {d:<35} n={count}")

print("\n" + "="*50)
print("UNIQUE PHYLA (non-prokaryotic):")
print("="*50)
for p in sorted(non_prok["Phylum"].dropna().unique()):
    count = (non_prok["Phylum"] == p).sum()
    print(f"  {p:<35} n={count}")

print("\n" + "="*50)
print("UNCLASSIFIED AT PHYLUM LEVEL:")
print("="*50)
unclass = non_prok["Phylum"].isna().sum()
print(f"  No phylum assignment: {unclass}")

print("\n" + "="*50)
print("FULL BREAKDOWN — DOMAIN x PHYLUM:")
print("="*50)
breakdown = non_prok.groupby(
    ["Domain", "Phylum"], dropna=False).size().reset_index(
    name="count")
print(breakdown.to_string(index=False))

Total ZOTUs: 6198

Non-prokaryotic ZOTUs: 302

UNIQUE DOMAINS (non-prokaryotic):
  eukaryota                           n=34
  fungi                               n=1
  metazoa                             n=3
  viridiplantae                       n=14

UNIQUE PHYLA (non-prokaryotic):
  bacillariophyta                     n=1
  basidiomycota                       n=1
  chlorophyta                         n=5
  chordata                            n=3
  ciliophora                          n=2
  discosea                            n=6
  eukaryota                           n=18
  eustigmatophyceae                   n=1
  oomycota                            n=1
  phaeophyceae                        n=1
  streptophyta                        n=9
  tubulinea                           n=4

UNCLASSIFIED AT PHYLUM LEVEL:
  No phylum assignment: 250

FULL BREAKDOWN — DOMAIN x PHYLUM:
       Domain            Phylum  count
    eukaryota   bacillariophyta      1
    eukaryota        ciliophora      2


In [5]:
#REMOVES EUKARYOTA

import pandas as pd


# ============================
# LOAD STANDARDIZED FILE
# ============================
path = "/content/zotutab_decontam_Final_standardized.csv"
df   = pd.read_csv(path)

print(f"Total ZOTUs before removal: {len(df)}")

# ============================
# NORMALISE
# ============================
df["Domain_norm"] = df["Domain"].str.strip().str.lower()
df["Phylum_norm"] = df["Phylum"].str.strip().str.lower()

# ============================
# EUKARYOTE IDENTIFIERS
# confirmed from output above
# ============================
eukaryote_domains = [
    "eukaryota", "fungi", "metazoa", "viridiplantae"
]

eukaryote_phyla = [
    "bacillariophyta", "basidiomycota", "chlorophyta",
    "chordata", "ciliophora", "discosea", "eukaryota",
    "eustigmatophyceae", "oomycota", "phaeophyceae",
    "streptophyta", "tubulinea",
]

# ============================
# FLAG EUKARYOTES
# remove by domain OR phylum OR unclassified non-prokaryote
# ============================
euk_by_domain = df["Domain_norm"].isin(eukaryote_domains)
euk_by_phylum = df["Phylum_norm"].isin(eukaryote_phyla)

# unclassified non-prokaryotes — NaN phylum and non-prokaryote domain
euk_unclassified = (
    df["Phylum_norm"].isna() &
    ~df["Domain_norm"].isin(["bacteria", "archaea"])
)

euk_mask = euk_by_domain | euk_by_phylum | euk_unclassified

# ============================
# AUDIT
# ============================
print(f"\nRemoval audit:")
print(f"  Flagged by domain:              {euk_by_domain.sum()}")
print(f"  Flagged by phylum:              {euk_by_phylum.sum()}")
print(f"  Flagged as unclassified non-prok: {euk_unclassified.sum()}")
print(f"  Total flagged:                  {euk_mask.sum()}")
print(f"  Expected removal:               302")
print(f"  Match: {euk_mask.sum() == 302}")

# ============================
# REMOVE AND CLEAN
# ============================
df_clean = df[~euk_mask].copy()
df_clean = df_clean.drop(columns=["Domain_norm", "Phylum_norm"])

# ============================
# VERIFY
# ============================
remaining_euk = df_clean["Domain"].str.strip().str.lower().isin(
    eukaryote_domains).sum()
remaining_prok = len(df_clean)

print(f"\nVerification:")
print(f"  ZOTUs remaining:              {remaining_prok}")
print(f"  Expected prokaryotic ZOTUs:   5896")
print(f"  Match: {remaining_prok == 5896}")
print(f"  Eukaryotic domains remaining: {remaining_euk} (should be 0)")

print(f"\nDomain breakdown after removal:")
print(df_clean["Domain"].str.lower().str.strip()
      .value_counts().to_string())

print(f"\nUnique phyla after removal: "
      f"{df_clean['Phylum'].nunique()}")

# ============================
# SAVE
# ============================
output_path = "/content/zotutab_decontam_Final_prokaryotes.csv"
df_clean.to_csv(output_path, index=False)
print(f"\nSaved: {output_path}")
print(f"Final shape: {df_clean.shape}")

Total ZOTUs before removal: 6198

Removal audit:
  Flagged by domain:              52
  Flagged by phylum:              52
  Flagged as unclassified non-prok: 250
  Total flagged:                  302
  Expected removal:               302
  Match: True

Verification:
  ZOTUs remaining:              5896
  Expected prokaryotic ZOTUs:   5896
  Match: True
  Eukaryotic domains remaining: 0 (should be 0)

Domain breakdown after removal:
Domain
bacteria    5547
archaea      349

Unique phyla after removal: 37

Saved: /content/zotutab_decontam_Final_prokaryotes.csv
Final shape: (5896, 70)


In [6]:
#A CHECK TO SEE ALL EUKARYOTES, UNCLASSIFIED PHYLA, AND OLD NAMES ARE REPLACED
import pandas as pd

# ============================
# LOAD FINAL PROKARYOTE FILE
# ============================
path = "/content/zotutab_decontam_Final_prokaryotes.csv"
df   = pd.read_csv(path)

print("=" * 60)
print("QUALITY CHECK — zotutab_decontam_Final_prokaryotes.csv")
print("=" * 60)

# ============================
# RENAME MAP — all old names
# that should NOT appear
# ============================
old_names = [
    'actinobacteria', 'firmicutes', 'bacteroidetes',
    'chloroflexi', 'cyanobacteria', 'proteobacteria',
    'pseudomonadot', 'planctomycetes', 'gemmatimonadetes',
    'verrucomicrobia', 'acidobacteria', 'nitrospirae',
    'chlamydiae', 'elusimicrobia', 'spirochaetes',
    'tenericutes', 'fusobacteria', 'thermodesulfobacteria',
    'deinococcus-thermus', 'armatimonadetes', 'thermotogae',
]

eukaryote_domains = [
    'eukaryota', 'fungi', 'metazoa', 'viridiplantae'
]

eukaryote_phyla = [
    'bacillariophyta', 'basidiomycota', 'chlorophyta',
    'chordata', 'ciliophora', 'discosea', 'eukaryota',
    'eustigmatophyceae', 'oomycota', 'phaeophyceae',
    'streptophyta', 'tubulinea',
]

# ============================
# NORMALISE FOR CHECKING
# ============================
df["Domain_norm"] = df["Domain"].str.strip().str.lower()
df["Phylum_norm"] = df["Phylum"].str.strip().str.lower()

# ============================
# CHECK 1 — TOTAL ZOTU COUNT
# expected: 5896
# ============================
print(f"\n{'CHECK 1 — TOTAL ZOTU COUNT':}")
print(f"  Total ZOTUs: {len(df)}")
print(f"  Expected:    5896")
print(f"  {'PASS' if len(df) == 5896 else 'FAIL'}")

# ============================
# CHECK 2 — NO BLANK SAMPLES
# ============================
print(f"\nCHECK 2 — NO BLANK SAMPLES")
blank_cols = [c for c in df.columns if "Blank" in str(c)
              or "blank" in str(c)]
print(f"  Blank columns found: {len(blank_cols)}")
if blank_cols:
    print(f"  FAIL — blanks present: {blank_cols}")
else:
    print(f"  PASS — no blank columns")

# ============================
# CHECK 3 — NO EUKARYOTES BY DOMAIN
# ============================
print(f"\nCHECK 3 — NO EUKARYOTIC DOMAINS")
euk_domain = df["Domain_norm"].isin(eukaryote_domains).sum()
print(f"  Eukaryotic domain rows: {euk_domain}")
print(f"  {'PASS' if euk_domain == 0 else 'FAIL'}")
if euk_domain > 0:
    print(f"  Domains found: "
          f"{df.loc[df['Domain_norm'].isin(eukaryote_domains), 'Domain'].unique()}")

# ============================
# CHECK 4 — NO EUKARYOTES BY PHYLUM
# ============================
print(f"\nCHECK 4 — NO EUKARYOTIC PHYLA")
euk_phylum = df["Phylum_norm"].isin(eukaryote_phyla).sum()
print(f"  Eukaryotic phylum rows: {euk_phylum}")
print(f"  {'PASS' if euk_phylum == 0 else 'FAIL'}")
if euk_phylum > 0:
    print(f"  Phyla found: "
          f"{df.loc[df['Phylum_norm'].isin(eukaryote_phyla), 'Phylum'].unique()}")

# ============================
# CHECK 5 — NO OLD PHYLUM NAMES
# ============================
print(f"\nCHECK 5 — NO OLD PHYLUM NAMES (rename map applied)")
old_found = df["Phylum_norm"].isin(old_names)
print(f"  Old names found: {old_found.sum()}")
print(f"  {'PASS' if old_found.sum() == 0 else 'FAIL'}")
if old_found.sum() > 0:
    print(f"  Old names still present:")
    for n in df.loc[old_found, "Phylum"].unique():
        print(f"    {n}")

# ============================
# CHECK 6 — NO UNCLASSIFIED PHYLA
# ============================
print(f"\nCHECK 6 — NO UNCLASSIFIED PHYLA")
unclass_phylum = df["Phylum_norm"].isna().sum()
unclass_str    = df["Phylum_norm"].str.startswith(
    "unclassified", na=False).sum()
print(f"  NaN phylum:            {unclass_phylum}")
print(f"  'unclassified' string: {unclass_str}")
total_unclass = unclass_phylum + unclass_str
print(f"  Total unclassified:    {total_unclass}")
print(f"  {'PASS' if total_unclass == 0 else 'FAIL'}")

# ============================
# CHECK 7 — ONLY BACTERIA AND ARCHAEA
# ============================
print(f"\nCHECK 7 — ONLY BACTERIA AND ARCHAEA")
domain_counts = df["Domain_norm"].value_counts()
print(f"  Domain breakdown:")
for d, n in domain_counts.items():
    print(f"    {d:<20} n={n}")
only_prok = set(domain_counts.index) <= {"bacteria", "archaea"}
print(f"  {'PASS' if only_prok else 'FAIL'}")

# ============================
# CHECK 8 — BACTERIA AND ARCHAEA COUNTS
# ============================
print(f"\nCHECK 8 — BACTERIA AND ARCHAEA COUNTS")
n_bact = (df["Domain_norm"] == "bacteria").sum()
n_arch = (df["Domain_norm"] == "archaea").sum()
print(f"  Bacteria: {n_bact}  (expected 5547)")
print(f"  Archaea:  {n_arch}  (expected 349)")
print(f"  Bacteria {'PASS' if n_bact == 5547 else 'FAIL'}")
print(f"  Archaea  {'PASS' if n_arch == 349  else 'FAIL'}")

# ============================
# CHECK 9 — UNIQUE PHYLA
# ============================
print(f"\nCHECK 9 — UNIQUE PHYLA COUNT")
n_phyla = df["Phylum_norm"].nunique()
print(f"  Unique phyla: {n_phyla}  (expected 36)")
print(f"  {'PASS' if n_phyla == 36 else 'FAIL'}")

print(f"\nAll unique phyla:")
for p in sorted(df["Phylum_norm"].dropna().unique()):
    n = (df["Phylum_norm"] == p).sum()
    print(f"  {p:<40} n={n}")

# ============================
# SUMMARY
# ============================
checks = {
    "Total ZOTUs = 5896":        len(df) == 5896,
    "No blank columns":          len(blank_cols) == 0,
    "No eukaryotic domains":     euk_domain == 0,
    "No eukaryotic phyla":       euk_phylum == 0,
    "No old phylum names":       old_found.sum() == 0,
    "No unclassified phyla":     total_unclass == 0,
    "Only bacteria and archaea": only_prok,
    "Bacteria count = 5547":     n_bact == 5547,
    "Archaea count = 349":       n_arch == 349,
    "Unique phyla = 36":         n_phyla == 36,
}

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
all_pass = True
for check, result in checks.items():
    status = "PASS" if result else "FAIL"
    if not result:
        all_pass = False
    print(f"  {status}  {check}")

print()
if all_pass:
    print("ALL CHECKS PASSED — file is clean and ready to use")
else:
    print("SOME CHECKS FAILED — review output above before proceeding")

# clean up normalisation columns
df.drop(columns=["Domain_norm", "Phylum_norm"], inplace=True)

QUALITY CHECK — zotutab_decontam_Final_prokaryotes.csv

CHECK 1 — TOTAL ZOTU COUNT
  Total ZOTUs: 5896
  Expected:    5896
  PASS

CHECK 2 — NO BLANK SAMPLES
  Blank columns found: 0
  PASS — no blank columns

CHECK 3 — NO EUKARYOTIC DOMAINS
  Eukaryotic domain rows: 0
  PASS

CHECK 4 — NO EUKARYOTIC PHYLA
  Eukaryotic phylum rows: 0
  PASS

CHECK 5 — NO OLD PHYLUM NAMES (rename map applied)
  Old names found: 0
  PASS

CHECK 6 — NO UNCLASSIFIED PHYLA
  NaN phylum:            0
  'unclassified' string: 0
  Total unclassified:    0
  PASS

CHECK 7 — ONLY BACTERIA AND ARCHAEA
  Domain breakdown:
    bacteria             n=5547
    archaea              n=349
  PASS

CHECK 8 — BACTERIA AND ARCHAEA COUNTS
  Bacteria: 5547  (expected 5547)
  Archaea:  349  (expected 349)
  Bacteria PASS
  Archaea  PASS

CHECK 9 — UNIQUE PHYLA COUNT
  Unique phyla: 37  (expected 36)
  FAIL

All unique phyla:
  abditibacteriota                         n=6
  acidobacteriota                          n=255
  acti

In [7]:
import pandas as pd
import csv
import io

# ============================
# PATHS
# ============================
prokaryotes_path = "/content/zotutab_decontam_Final_prokaryotes.csv"
ref_path         = "/content/zotutab_decontam_unstandardized_naming.csv"
output_path      = "/content/zotutab_decontam_Final.csv"

# ============================
# STEP 1 — LOAD PROKARYOTE FILE
# remove ZOTUs where Phylum = 'bacteria'
# ============================
df = pd.read_csv(prokaryotes_path)
print(f"ZOTUs loaded: {len(df)}")

df["Phylum_norm"] = df["Phylum"].str.strip().str.lower()

problem = df[df["Phylum_norm"] == "bacteria"]
print(f"ZOTUs with Phylum = 'bacteria' (removing): {len(problem)}")

df_clean = df[df["Phylum_norm"] != "bacteria"].copy()
df_clean = df_clean.drop(columns=["Phylum_norm"])

print(f"ZOTUs after removal: {len(df_clean)}")
print(f"Expected 5892: {'PASS' if len(df_clean) == 5892 else 'FAIL'}")

n_phyla = df_clean["Phylum"].str.strip().str.lower().nunique()
print(f"Unique phyla: {n_phyla}  expected 36: "
      f"{'PASS' if n_phyla == 36 else 'FAIL'}")

# ============================
# STEP 2 — GET ANATOMY ROW FROM REFERENCE
# align to target column order
# ============================
with open(ref_path, "r") as f:
    ref_lines = f.readlines()

ref_anatomy = next(csv.reader(io.StringIO(ref_lines[0])))
ref_header  = next(csv.reader(io.StringIO(ref_lines[1])))

# map column name -> anatomy value from reference
ref_col_to_anat = {
    col: anat for col, anat
    in zip(ref_header, ref_anatomy)
}

# build anatomy row aligned to df_clean columns
aligned_anatomy = [
    ref_col_to_anat.get(col, "")
    for col in df_clean.columns
]

print(f"\nAnatomy alignment check:")
print(f"  df_clean columns:    {len(df_clean.columns)}")
print(f"  Aligned anatomy:     {len(aligned_anatomy)}")
print(f"  First 5 columns:     {list(df_clean.columns[:5])}")
print(f"  First 5 anatomy:     {aligned_anatomy[:5]}")
print(f"  Last 5 columns:      {list(df_clean.columns[-5:])}")
print(f"  Last 5 anatomy:      {aligned_anatomy[-5:]}")

# ============================
# STEP 3 — WRITE FINAL FILE
# row 1 = anatomy
# row 2 = column headers
# row 3+ = data
# ============================
with open(output_path, "w", newline="") as f:
    writer = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
    writer.writerow(aligned_anatomy)

# append header + data from dataframe
df_clean.to_csv(output_path, mode="a", index=False)

print(f"\nSaved: {output_path}")

# ============================
# STEP 4 — VERIFY
# ============================
with open(output_path, "r") as f:
    verify_lines = f.readlines()

anat_cols = next(csv.reader(io.StringIO(verify_lines[0])))
head_cols = next(csv.reader(io.StringIO(verify_lines[1])))
data_cols = next(csv.reader(io.StringIO(verify_lines[2])))

print(f"\nVerification:")
print(f"  Total lines:   {len(verify_lines)}  "
      f"(expected 5894 = 1 anatomy + 1 header + 5892 data)")
print(f"  Anatomy cols:  {len(anat_cols)}")
print(f"  Header cols:   {len(head_cols)}")
print(f"  Data cols:     {len(data_cols)}")
print(f"  All match:     "
      f"{len(anat_cols) == len(head_cols) == len(data_cols)}")

print(f"\nLine 1 (anatomy, first 150 chars):")
print(verify_lines[0][:150])
print(f"\nLine 2 (header, first 150 chars):")
print(verify_lines[1][:150])
print(f"\nLine 3 (first data row, first 150 chars):")
print(verify_lines[2][:150])

print(f"\nExpected 5894 lines: "
      f"{'PASS' if len(verify_lines) == 5894 else 'FAIL'}")
print(f"Columns match: "
      f"{'PASS' if len(anat_cols) == len(head_cols) == len(data_cols) else 'FAIL'}")
print(f"Row 1 starts with ANATOMY: "
      f"{'PASS' if anat_cols[0] == 'ANATOMY' else 'FAIL'}")
print(f"Row 2 starts with Zotu: "
      f"{'PASS' if head_cols[0] == 'Zotu' else 'FAIL'}")

ZOTUs loaded: 5896
ZOTUs with Phylum = 'bacteria' (removing): 4
ZOTUs after removal: 5892
Expected 5892: PASS
Unique phyla: 36  expected 36: PASS

Anatomy alignment check:
  df_clean columns:    70
  Aligned anatomy:     70
  First 5 columns:     ['Zotu', 'MF.4B', 'MF.5B', 'MF.6B', 'MF.7B']
  First 5 anatomy:     ['ANATOMY', 'Interdune', 'Stoss', 'Lee', 'Interdune']
  Last 5 columns:      ['Class', 'Order', 'Family', 'Genus', 'Species']
  Last 5 anatomy:      ['', '', '', '', '']

Saved: /content/zotutab_decontam_Final.csv

Verification:
  Total lines:   5894  (expected 5894 = 1 anatomy + 1 header + 5892 data)
  Anatomy cols:  70
  Header cols:   70
  Data cols:     70
  All match:     True

Line 1 (anatomy, first 150 chars):
ANATOMY,Interdune,Stoss,Lee,Interdune,Interdune,Interdune,Interdune,Stoss,Lee,Interdune,Stoss,Lee,Interdune,Stoss,Lee,Interdune,"Interdune, Depth",Lee

Line 2 (header, first 150 chars):
Zotu,MF.4B,MF.5B,MF.6B,MF.7B,MF.8B,MF.11A,MF.12A,MF.13A,MF.14B,MF.15A,MF.16A,M

In [8]:
#MAKES RELATIVE ABUNDANCE
import pandas as pd

# ============================
# LOAD COUNT FILE
# ============================
ref_path = "/content/zotutab_decontam_unstandardized_naming.csv"
path     = "/content/zotutab_decontam_Final.csv"

# read anatomy line from reference
with open(ref_path, "r") as f:
    anatomy_line = f.readline()

# read count file — skip anatomy row
df = pd.read_csv(path, skiprows=1)

print(f"Loaded: {df.shape[0]} ZOTUs x {df.shape[1]} columns")

# ============================
# IDENTIFY COLUMNS
# ============================
tax_cols = ["Domain", "Phylum", "Class", "Order",
            "Family", "Genus", "Species"]
zotu_col = df.columns[0]

sample_cols = [c for c in df.columns
               if c not in tax_cols
               and c != zotu_col
               and "Blank" not in str(c)]

print(f"ZOTU column:    {zotu_col}")
print(f"Sample columns: {len(sample_cols)}")
print(f"First 3:        {sample_cols[:3]}")

# ============================
# CONVERT TO NUMERIC
# ============================
df[sample_cols] = df[sample_cols].apply(
    pd.to_numeric, errors="coerce").fillna(0)

# ============================
# CALCULATE RELATIVE ABUNDANCE
# ============================
col_sums  = df[sample_cols].sum(axis=0)
rel_abund = df[sample_cols].div(col_sums, axis=1) * 100

# ============================
# VERIFY
# ============================
col_sums_check = rel_abund.sum(axis=0)
print(f"\nVerification:")
print(f"  Min sum:  {col_sums_check.min():.4f}%  (should be 100)")
print(f"  Max sum:  {col_sums_check.max():.4f}%  (should be 100)")
print(f"  All 100%: {(col_sums_check.round(6) == 100).all()}")

# ============================
# BUILD OUTPUT IN CORRECT COLUMN ORDER
# Zotu | sample cols | taxonomy cols
# matches reference file structure exactly
# ============================
df_rel = pd.concat([
    df[[zotu_col]],        # ZOTU ID first
    rel_abund,             # sample columns with rel abund
    df[tax_cols]           # taxonomy columns at the end
], axis=1)

print(f"\nOutput shape: {df_rel.shape}")
print(f"Column order check:")
print(f"  First col:  {df_rel.columns[0]}  (should be Zotu)")
print(f"  Col 2:      {df_rel.columns[1]}  (should be first sample)")
print(f"  Last 7:     {df_rel.columns[-7:].tolist()}  (should be taxonomy)")

# spot check
test_sample = sample_cols[0]
print(f"\nSpot check — {test_sample}:")
print(f"  Rel abund sum: {rel_abund[test_sample].sum():.4f}%")
top3 = rel_abund[test_sample].nlargest(3)
for zotu, val in top3.items():
    print(f"  {zotu}: {val:.4f}%")

# ============================
# SAVE WITH ANATOMY ROW FIRST
# ============================
output_path = "/content/zotutab_decontam_RelAbund_Final.csv"

with open(output_path, "w") as f:
    f.write(anatomy_line)

df_rel.to_csv(output_path, mode="a", index=False)

print(f"\nSaved: {output_path}")

# ============================
# VERIFY FILE STRUCTURE
# ============================
with open(output_path, "r") as f:
    lines = f.readlines()

print(f"\nFile structure verification:")
print(f"  Total lines: {len(lines)}")
print(f"\nLine 1 (anatomy, first 150 chars):")
print(lines[0][:150])
print(f"\nLine 2 (headers, first 150 chars):")
print(lines[1][:150])
print(f"\nLine 3 (first data row, first 150 chars):")
print(lines[2][:150])

Loaded: 5892 ZOTUs x 70 columns
ZOTU column:    Zotu
Sample columns: 62
First 3:        ['MF.4B', 'MF.5B', 'MF.6B']

Verification:
  Min sum:  100.0000%  (should be 100)
  Max sum:  100.0000%  (should be 100)
  All 100%: True

Output shape: (5892, 70)
Column order check:
  First col:  Zotu  (should be Zotu)
  Col 2:      MF.4B  (should be first sample)
  Last 7:     ['Domain', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species']  (should be taxonomy)

Spot check — MF.4B:
  Rel abund sum: 100.0000%
  2: 19.0742%
  1: 13.4085%
  7: 3.4918%

Saved: /content/zotutab_decontam_RelAbund_Final.csv

File structure verification:
  Total lines: 5894

Line 1 (anatomy, first 150 chars):
ANATOMY,Interdune,Stoss,Lee,Interdune,Interdune,Interdune,Interdune,Stoss,Lee,Interdune,Stoss,Lee,Interdune,Stoss,Lee,Interdune,"Interdune, Depth",Lee

Line 2 (headers, first 150 chars):
Zotu,MF.4B,MF.5B,MF.6B,MF.7B,MF.8B,MF.11A,MF.12A,MF.13A,MF.14B,MF.15A,MF.16A,MF.17A,MF.18B,MF.19A,MF.20A,MF.27B,MF.28A,MF.30A

In [9]:
#CHECKS RARE AND DOMINANT SPLIT (MORE ON THIS IN LATER IPYNBS)
import pandas as pd

# ============================
# LOAD FINAL RELATIVE ABUNDANCE FILE
# ============================
path = "/content/zotutab_decontam_RelAbund_Final.csv"
df   = pd.read_csv(path)

print(f"Total ZOTUs: {len(df)}")

# ============================
# IDENTIFY SAMPLE COLUMNS
# ============================
tax_cols = ["Domain", "Phylum", "Class", "Order",
            "Family", "Genus", "Species"]
zotu_col = df.columns[0]

sample_cols = [c for c in df.columns
               if c not in tax_cols
               and c != zotu_col
               and "Blank" not in str(c)]

print(f"Sample columns: {len(sample_cols)}")

# ============================
# CONVERT TO NUMERIC
# ============================
df[sample_cols] = df[sample_cols].apply(
    pd.to_numeric, errors="coerce").fillna(0)

# ============================
# RARE / DOMINANT SPLIT
# rare     = never exceeds 1% in any single sample
# dominant = exceeds 1% in at least one sample
# ============================
zotu_max = df[sample_cols].max(axis=1)

rare_mask     = zotu_max <= 1.0
dominant_mask = zotu_max >  1.0

n_rare     = rare_mask.sum()
n_dominant = dominant_mask.sum()
n_total    = len(df)

print("\n" + "="*55)
print("RARE / DOMINANT SPLIT")
print("(threshold: >1% max relative abundance in any sample)")
print("="*55)
print(f"Total ZOTUs:                        {n_total}")
print(f"Rare ZOTUs (≤1% in all samples):   {n_rare} "
      f"({n_rare/n_total*100:.1f}%)")
print(f"Dominant ZOTUs (>1% in ≥1 sample): {n_dominant} "
      f"({n_dominant/n_total*100:.1f}%)")
print(f"Check rare + dominant = total:      "
      f"{n_rare + n_dominant == n_total}")

print(f"\nExpected from text:")
print(f"  Rare:     5,594  (94.9%)")
print(f"  Dominant: 302    (5.1%)")
print(f"  Rare match:     {n_rare == 5594}")
print(f"  Dominant match: {n_dominant == 302}")

# ============================
# RELATIVE ABUNDANCE CHECK
# ============================
total_relabund    = df[sample_cols].sum().sum()
dominant_relabund = df.loc[dominant_mask, sample_cols].sum().sum()
rare_relabund     = df.loc[rare_mask,     sample_cols].sum().sum()

print(f"\nRelative abundance accounted for:")
print(f"  Dominant ZOTUs: "
      f"{dominant_relabund/total_relabund*100:.1f}% of total")
print(f"  Rare ZOTUs:     "
      f"{rare_relabund/total_relabund*100:.1f}% of total")

Total ZOTUs: 5893
Sample columns: 69

RARE / DOMINANT SPLIT
(threshold: >1% max relative abundance in any sample)
Total ZOTUs:                        5893
Rare ZOTUs (≤1% in all samples):   5566 (94.5%)
Dominant ZOTUs (>1% in ≥1 sample): 327 (5.5%)
Check rare + dominant = total:      True

Expected from text:
  Rare:     5,594  (94.9%)
  Dominant: 302    (5.1%)
  Rare match:     False
  Dominant match: False

Relative abundance accounted for:
  Dominant ZOTUs: 52.3% of total
  Rare ZOTUs:     47.7% of total


In [12]:
import pandas as pd

# ============================
# FUNCTION TO SPLIT FILE INTO
# SURFACE AND DEPTH VERSIONS
# ============================
def split_surface_depth(input_path, output_surface_path,
                         output_depth_path, label):

    print(f"\n{'='*55}")
    print(f"Processing: {label}")
    print(f"{'='*55}")

    # read raw preserving two header rows
    with open(input_path, "r") as f:
        lines = f.readlines()

    anatomy_line = lines[0]
    header_line  = lines[1]
    data_lines   = lines[2:]

    print(f"Anatomy line (first 100 chars): {anatomy_line[:100]}")
    print(f"Header line  (first 100 chars): {header_line[:100]}")
    print(f"Data rows: {len(data_lines)}")

    # parse anatomy and header rows
    # handle quoted fields with commas
    import csv
    import io

    anatomy_vals = next(csv.reader(io.StringIO(anatomy_line)))
    header_vals  = next(csv.reader(io.StringIO(header_line)))

    print(f"\nTotal columns: {len(header_vals)}")
    print(f"First 5 headers: {header_vals[:5]}")

    # ============================
    # IDENTIFY SURFACE VS DEPTH COLUMNS
    # ============================
    tax_cols = ["Domain", "Phylum", "Class", "Order",
                "Family", "Genus", "Species"]
    zotu_col = header_vals[0]

    surface_col_indices = []
    depth_col_indices   = []
    tax_col_indices     = []
    zotu_col_index      = [0]

    for i, (head, anat) in enumerate(
            zip(header_vals, anatomy_vals)):
        if i == 0:
            continue  # zotu col
        if head in tax_cols:
            tax_col_indices.append(i)
        elif "Depth" in str(anat):
            depth_col_indices.append(i)
        elif "Blank" in str(head):
            pass  # skip blanks
        else:
            surface_col_indices.append(i)

    # all non-zotu non-tax indices
    surface_names = [header_vals[i] for i in surface_col_indices]
    depth_names   = [header_vals[i] for i in depth_col_indices]

    print(f"\nSurface samples: {len(surface_names)}")
    print(f"Depth samples:   {len(depth_names)}")
    print(f"Depth samples removed: {depth_names}")

    # ============================
    # BUILD COLUMN INDICES FOR EACH OUTPUT
    # zotu + surface samples + taxonomy
    # zotu + depth samples + taxonomy
    # ============================
    surface_indices = [0] + surface_col_indices + tax_col_indices
    depth_indices   = [0] + depth_col_indices   + tax_col_indices

    def extract_cols(row_vals, indices):
        return [row_vals[i] if i < len(row_vals)
                else "" for i in indices]

    def write_csv(output_path, col_indices,
                  anatomy_vals, header_vals, data_lines):

        anatomy_out = extract_cols(anatomy_vals, col_indices)
        header_out  = extract_cols(header_vals,  col_indices)

        with open(output_path, "w", newline="") as f:
            writer = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
            writer.writerow(anatomy_out)
            writer.writerow(header_out)
            for line in data_lines:
                row = next(csv.reader(io.StringIO(line)))
                writer.writerow(extract_cols(row, col_indices))

        print(f"\nSaved: {output_path}")

        # verify
        with open(output_path, "r") as f:
            vlines = f.readlines()
        print(f"  Total lines: {len(vlines)}")
        print(f"  Line 1 (first 80): {vlines[0][:80]}")
        print(f"  Line 2 (first 80): {vlines[1][:80]}")
        print(f"  Line 3 (first 80): {vlines[2][:80]}")

    # write surface file
    write_csv(output_surface_path, surface_indices,
              anatomy_vals, header_vals, data_lines)

    # write depth file
    write_csv(output_depth_path, depth_indices,
              anatomy_vals, header_vals, data_lines)

    return surface_names, depth_names


# ============================
# PROCESS COUNT FILE
# ============================
surf_samples, depth_samples = split_surface_depth(
    input_path          = "/content/zotutab_decontam_Final.csv",
    output_surface_path = "/content/zotutab_decontam_Final.csv",
    output_depth_path   = "/content/zotutab_decontam_Final_Depth_Only.csv",
    label               = "COUNT FILE"
)

# ============================
# PROCESS RELABUND FILE
# ============================
split_surface_depth(
    input_path          = "/content/zotutab_decontam_RelAbund_Final.csv",
    output_surface_path = "/content/zotutab_decontam_RelAbund_Final.csv",
    output_depth_path   = "/content/zotutab_decontam_RelAbund_Final_Depth_Only.csv",
    label               = "RELATIVE ABUNDANCE FILE"
)

# ============================
# FINAL SUMMARY
# ============================
print("\n" + "="*55)
print("SUMMARY")
print("="*55)
print(f"Surface samples retained: {len(surf_samples)}")
print(f"Depth samples moved:      {len(depth_samples)}")
print(f"\nDepth samples:")
for s in depth_samples:
    print(f"  {s}")
print(f"\nFiles updated:")
print(f"  /content/zotutab_decontam_Final.csv")
print(f"  /content/zotutab_decontam_RelAbund_Final.csv")
print(f"\nDepth files created:")
print(f"  /content/zotutab_decontam_Final_Depth_Only.csv")
print(f"  /content/zotutab_decontam_RelAbund_Final_Depth_Only.csv")


Processing: COUNT FILE
Anatomy line (first 100 chars): ANATOMY,Interdune,Stoss,Lee,Interdune,Interdune,Interdune,Interdune,Stoss,Lee,Interdune,Stoss,Lee,In
Header line  (first 100 chars): Zotu,MF.4B,MF.5B,MF.6B,MF.7B,MF.8B,MF.11A,MF.12A,MF.13A,MF.14B,MF.15A,MF.16A,MF.17A,MF.18B,MF.19A,MF
Data rows: 5892

Total columns: 60
First 5 headers: ['Zotu', 'MF.4B', 'MF.5B', 'MF.6B', 'MF.7B']

Surface samples: 52
Depth samples:   0
Depth samples removed: []

Saved: /content/zotutab_decontam_Final.csv
  Total lines: 5894
  Line 1 (first 80): ANATOMY,Interdune,Stoss,Lee,Interdune,Interdune,Interdune,Interdune,Stoss,Lee,In
  Line 2 (first 80): Zotu,MF.4B,MF.5B,MF.6B,MF.7B,MF.8B,MF.11A,MF.12A,MF.13A,MF.14B,MF.15A,MF.16A,MF.
  Line 3 (first 80): Zotu5,384.0,393.0,169.0,161.0,41.0,39.0,333.0,339.0,481.0,1176.0,414.0,283.0,34.

Saved: /content/zotutab_decontam_Final_Depth_Only.csv
  Total lines: 5894
  Line 1 (first 80): ANATOMY,,,,,,,

  Line 2 (first 80): Zotu,Domain,Phylum,Class,Order,Family,Genus,